# D-MTHD tweet benchmark

Right panel: Accelerator **GPU T4 x2**, Internet **On**.

**This is a multi-version run.** Kaggle stops a session at 12 hours, which is less than the full
grid needs. Each version does as much as it can, then stops itself at 11 hours so the packaging
cell below still runs. To continue: add this version's output as an input (Add Input -> Your Work)
and Save & Run All again. The resume path is detected automatically and finished work is skipped.

The committee here includes the **implicit-abuse specialist**, the only teacher that has seen
abuse-by-implication labelled as such. This notebook builds it on first use: it downloads the two
implicit corpora, trains HateBERT on them, and also trains the headline student on them as the
data-versus-distillation control. That costs roughly an hour once, lands in
`runs/implicit/specialist`, and is reused by every later version you attach. `SPECIALIST=0` skips it.

Download `dmthd_tweets_results.tgz` from the output: it holds the metrics, histories, predictions
and generated tables, without the model weights, so it is small.


In [ ]:
import os, subprocess, glob, shutil, zipfile
REPO = "https://github.com/mahdihasanshadi/THESIS.git"
DEST = "/kaggle/working/dmthd-p3"
if not os.path.exists(os.path.join(DEST, "src", "dmthd")):
    r = subprocess.run(["git", "clone", "-q", REPO, DEST])          # works if the repo is public
    if r.returncode != 0:
        shutil.rmtree(DEST, ignore_errors=True)
        z = glob.glob("/kaggle/input/**/dmthd-p3-code.zip", recursive=True)
        tree = glob.glob("/kaggle/input/**/src/dmthd/train_student.py", recursive=True)
        if z:                                                            # zip attached as-is
            zipfile.ZipFile(z[0]).extractall(DEST)
        elif tree:                                                       # Kaggle auto-extracted the zip
            root = os.path.dirname(os.path.dirname(os.path.dirname(tree[0])))
            shutil.copytree(root, DEST)
        else:
            raise SystemExit("clone failed and no code found among the inputs: attach the code dataset")
os.chdir(DEST)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
print(open("README.md").read()[:400])

In [ ]:
import os, subprocess, glob, sys, urllib.request
env = dict(os.environ, ROOT='/kaggle/working', PYTHONPATH='src', GPU='1', SEEDS='1,2,3',
           COMMITTEES='homo,hetero', MODES='ft,skd,uniform,dmthd',
           TEACHERS='bert-large-uncased:bert-large,GroNLP/hateBERT:hatebert,cardiffnlp/twitter-roberta-base-irony:irony',
           STUDENTS='google/bert_uncased_L-4_H-256_A-4:bert-mini,google/bert_uncased_L-4_H-512_A-8:bert-small,distilbert-base-uncased:distilbert')
# resume: find a previous notebook output among the attached inputs (Kaggle mounts them at
# /kaggle/input/notebooks/<user>/<slug>, so the path is detected rather than typed)
env['TIME_BUDGET_S'] = '39600'   # stop launching work after 11 h so the packaging cell below still runs
env['ABLATION_SEEDS'] = '1'      # ablations are supporting evidence: one seed, stated in Limitations
env['IMPLICIT_RAW'] = '/kaggle/working/raw'   # the implicit corpora are downloaded from the Hub
# The implicit-abuse specialist joins this committee: HateBERT trained on the implicit benchmark,
# then task-adapted here like any other teacher. If a previous notebook output with
# runs/implicit/specialist is attached it is reused; otherwise it is trained once, which costs about
# 40 minutes and is why the implicit notebook is worth running first. SPECIALIST=0 turns it off.
env['SPECIALIST'] = os.environ.get('SPECIALIST', '1')
cands = sorted({p.replace(chr(92), '/').split('/runs/')[0]
                for pat in ('/kaggle/input/**/runs/tweets', '/kaggle/input/**/runs/implicit/specialist')
                for p in glob.glob(pat, recursive=True)})
env['RESUME_FROM'] = ','.join(cands)   # every attached previous output is merged, richest last
print('resume sources:', cands or '(none: starting fresh)', flush=True)
raw = (glob.glob('/kaggle/input/**/cyberbullying_tweets.csv', recursive=True) or [None])[0]
print('inputs seen:', glob.glob('/kaggle/input/*'), '| csv from input:', raw, flush=True)
if raw is None:   # dataset not attached: fetch the same file from its Hugging Face mirror (Internet must be on)
    os.makedirs('/kaggle/working/raw', exist_ok=True)
    raw = '/kaggle/working/raw/cyberbullying_tweets.csv'
    urllib.request.urlretrieve('https://huggingface.co/datasets/mattematics/cyberbullying/resolve/main/cyberbullying_tweets.csv', raw)
    print('downloaded corpus from Hugging Face:', os.path.getsize(raw), 'bytes', flush=True)
cmd = ['python', 'kaggle/run_benchmark.py', '--dataset', 'tweets', '--stage', 'all', '--raw', raw]
print('running:', ' '.join(cmd), flush=True)
r = subprocess.run(cmd, env=env)
if r.returncode != 0:
    raise SystemExit(f'BENCHMARK FAILED with exit code {r.returncode}: scroll up in this log to the first Traceback')
print('BENCHMARK FINISHED')


In [ ]:
# Package two things: a small results-only archive (metrics, histories, predictions, tables) that is
# easy to download, and the tables themselves. Model weights stay in /kaggle/working for the next
# version to resume from; they are far too large to download.
import glob, os, subprocess, tarfile
ROOT, DS = '/kaggle/working', 'tweets'
keep = []
for pat in ('results.json', 'eval_*.json', 'history.csv', 'test_probs.npy', 'test_labels.npy',
            'quantize_eval.json', 'transfer_*.json', 'dropped_teachers.json'):
    keep += glob.glob(f'{ROOT}/runs/{DS}/**/{pat}', recursive=True)
keep += glob.glob(f'{ROOT}/runs/{DS}/*.csv') + glob.glob(f'{ROOT}/cache/{DS}/meta.json') + glob.glob(f'{ROOT}/data/{DS}/report.json')
subprocess.run(['python', '-m', 'dmthd.tables', '--runs', f'{ROOT}/runs/{DS}', '--data', f'{ROOT}/data/{DS}',
                '--out', f'{ROOT}/tables_{DS}'], env=dict(os.environ, PYTHONPATH='src'))
keep += glob.glob(f'{ROOT}/tables_{DS}/*')
out = f'{ROOT}/dmthd_{DS}_results.tgz'
with tarfile.open(out, 'w:gz') as t:
    for f in keep:
        t.add(f, arcname=os.path.relpath(f, ROOT))
print(f'{len(keep)} files -> {out} ({os.path.getsize(out) / 2**20:.1f} MB) : download this one')
print('finished runs:', len(glob.glob(f'{ROOT}/runs/{DS}/*/*/seed*/results.json')))
